In [ ]:
import os
import warnings
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import datasets, transforms
from sklearn.decomposition import PCA
from PIL import Image
from tqdm import tqdm

warnings.filterwarnings('ignore')
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
os.makedirs('data', exist_ok=True)


In [ ]:
transform_mnist = transforms.Compose([transforms.ToTensor()])

train_mnist = datasets.MNIST('data', train=True,  download=True, transform=transform_mnist)
test_mnist  = datasets.MNIST('data', train=False, download=True, transform=transform_mnist)

train_loader = DataLoader(train_mnist, batch_size=128, shuffle=True,  pin_memory=True, num_workers=0)
test_loader  = DataLoader(test_mnist,  batch_size=128, shuffle=False, pin_memory=True, num_workers=0)

print(f'Train: {len(train_mnist):,} | Test: {len(test_mnist):,} | Shape: {train_mnist[0][0].shape}')


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i in range(10):
    img, label = train_mnist[i]
    axes[i // 5, i % 5].imshow(img.squeeze(), cmap='gray')
    axes[i // 5, i % 5].set_title(f'Label: {label}')
    axes[i // 5, i % 5].axis('off')
plt.suptitle('MNIST Sample Images')
plt.tight_layout()
plt.show()


## Section 1: Autoencoders

In [ ]:
class ConvAutoencoder(nn.Module):
    """Convolutional autoencoder for 28x28 grayscale images.

    Encoder: 1x28x28 → conv-pool x2 → FC → latent_dim
    Decoder: latent_dim → FC → deconv x2 → 1x28x28  (Sigmoid output in [0,1])
    """
    def __init__(self, latent_dim=32):
        super().__init__()
        self.latent_dim = latent_dim

        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1),     # → 32×14×14
            nn.BatchNorm2d(32), nn.LeakyReLU(0.2),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),    # → 64×7×7
            nn.BatchNorm2d(64), nn.LeakyReLU(0.2),
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 256), nn.LeakyReLU(0.2),
            nn.Linear(256, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256), nn.LeakyReLU(0.2),
            nn.Linear(256, 64 * 7 * 7), nn.LeakyReLU(0.2),
            nn.Unflatten(1, (64, 7, 7)),
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1),  # → 32×14×14
            nn.BatchNorm2d(32), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(32,  1, 3, stride=2, padding=1, output_padding=1),  # →  1×28×28
            nn.Sigmoid(),
        )

    def encode(self, x): return self.encoder(x)
    def decode(self, z): return self.decoder(z)
    def forward(self, x):
        z = self.encode(x)
        return self.decode(z), z


ae = ConvAutoencoder(latent_dim=32).to(device)
n_params = sum(p.numel() for p in ae.parameters())
print(f'AE parameters: {n_params:,}')
print(f'Compression:   784 → {ae.latent_dim}  ({(1 - ae.latent_dim/784)*100:.1f}% reduction)')

# Smoke-test forward pass
_x = torch.randn(4, 1, 28, 28).to(device)
_r, _z = ae(_x)
print(f'Input: {_x.shape} | Latent: {_z.shape} | Output: {_r.shape}')
del _x, _r, _z


In [ ]:
def train_ae(model, train_loader, test_loader, num_epochs=30):
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    train_losses, val_losses = [], []

    for epoch in range(num_epochs):
        model.train()
        t_loss = 0.0
        for data, _ in train_loader:
            data = data.to(device)
            recon, _ = model(data)
            loss = F.mse_loss(recon, data)
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            t_loss += loss.item()

        model.eval()
        v_loss = 0.0
        with torch.no_grad():
            for data, _ in test_loader:
                data = data.to(device)
                recon, _ = model(data)
                v_loss += F.mse_loss(recon, data).item()

        scheduler.step()
        train_losses.append(t_loss / len(train_loader))
        val_losses.append(v_loss / len(test_loader))

        if (epoch + 1) % 10 == 0:
            print(f'Epoch {epoch+1:3d}/{num_epochs} | '
                  f'Train: {train_losses[-1]:.5f} | Val: {val_losses[-1]:.5f}')

    return train_losses, val_losses


ae_train_losses, ae_val_losses = train_ae(ae, train_loader, test_loader)

plt.figure(figsize=(8, 4))
plt.plot(ae_train_losses, label='Train')
plt.plot(ae_val_losses,   label='Validation')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Autoencoder Training Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
ae.eval()
sample_imgs, sample_labels, ae_recons, ae_latents = [], [], [], []

with torch.no_grad():
    for data, labels in test_loader:
        data = data.to(device)
        recon, z = ae(data)
        sample_imgs.append(data.cpu())
        ae_recons.append(recon.cpu())
        ae_latents.append(z.cpu())
        sample_labels.append(labels)
        if sum(len(b) for b in sample_imgs) >= 512:
            break

sample_imgs   = torch.cat(sample_imgs)[:512]
ae_recons     = torch.cat(ae_recons)[:512]
ae_latents    = torch.cat(ae_latents)[:512]
sample_labels = torch.cat(sample_labels)[:512]

n = 10
fig, axes = plt.subplots(2, n, figsize=(16, 4))
for i in range(n):
    axes[0, i].imshow(sample_imgs[i].squeeze(), cmap='gray')
    axes[0, i].axis('off')
    axes[1, i].imshow(ae_recons[i].squeeze(), cmap='gray')
    axes[1, i].axis('off')
axes[0, 0].set_ylabel('Original', fontsize=11)
axes[1, 0].set_ylabel('AE Recon',  fontsize=11)
plt.suptitle('Autoencoder: Test-Set Reconstruction')
plt.tight_layout()
plt.show()

mse  = F.mse_loss(ae_recons, sample_imgs).item()
psnr = 10 * np.log10(1.0 / mse)
print(f'Reconstruction MSE:  {mse:.5f}')
print(f'Reconstruction PSNR: {psnr:.2f} dB')


In [ ]:
pca = PCA(n_components=2)
latent_2d = pca.fit_transform(ae_latents.numpy())

z_range = np.linspace(-4, 4, 300)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PCA scatter coloured by digit class
for c in range(10):
    mask = sample_labels == c
    axes[0].scatter(latent_2d[mask, 0], latent_2d[mask, 1],
                    label=str(c), alpha=0.5, s=8)
axes[0].set_title('AE Latent Space (PCA)')
axes[0].legend(markerscale=2, loc='best', title='Digit')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')

# Marginal distribution vs N(0,1)
axes[1].hist(ae_latents.numpy().flatten(), bins=100, density=True,
             alpha=0.7, label='AE latents')
axes[1].plot(z_range, np.exp(-0.5 * z_range**2) / np.sqrt(2 * np.pi),
             'r--', lw=2, label='N(0,1)')
axes[1].set_title('AE Latent Distribution — Unregularized')
axes[1].set_xlabel('Value')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'AE latent | mean: {ae_latents.mean():.3f}  std: {ae_latents.std():.3f}')
print('The distribution is not regularized → random sampling from N(0,1) will fail.')


In [ ]:
ae.eval()

with torch.no_grad():
    # Random sampling from N(0,I) — the latent space is not aligned with this prior
    rand_z    = torch.randn(16, ae.latent_dim).to(device)
    rand_imgs = ae.decode(rand_z).cpu()

    # Latent interpolation between two real images
    z_a        = ae.encode(sample_imgs[0:1].to(device))
    z_b        = ae.encode(sample_imgs[4:5].to(device))
    alphas     = torch.linspace(0, 1, 8)
    interp_z   = torch.stack([(1 - a) * z_a + a * z_b for a in alphas]).squeeze(1)
    interp_ae  = ae.decode(interp_z).cpu()

# Random-sample grid
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i in range(16):
    axes[i // 8, i % 8].imshow(rand_imgs[i].squeeze(), cmap='gray')
    axes[i // 8, i % 8].axis('off')
plt.suptitle('AE: Random Samples from N(0,I)  →  Incoherent '
             '(Latent Space Unregularized)')
plt.tight_layout()
plt.show()

# Interpolation strip
fig, axes = plt.subplots(1, 10, figsize=(15, 2))
axes[0].imshow(sample_imgs[0].squeeze(), cmap='gray')
axes[0].set_title('Start')
axes[0].axis('off')
for i, img in enumerate(interp_ae):
    axes[i + 1].imshow(img.squeeze(), cmap='gray')
    axes[i + 1].set_title(f'{alphas[i]:.2f}')
    axes[i + 1].axis('off')
axes[9].imshow(sample_imgs[4].squeeze(), cmap='gray')
axes[9].set_title('End')
axes[9].axis('off')
plt.suptitle('AE Latent Interpolation — Abrupt / Non-smooth Transitions')
plt.tight_layout()
plt.show()


## Section 2: Variational Autoencoders

In [ ]:
class ConvVAE(nn.Module):
    """Convolutional VAE for 28x28 grayscale images.

    The encoder outputs (mu, logvar) parameterising q(z|x) ~ N(mu, diag(exp(logvar))).
    The reparameterisation trick allows gradients to flow through the sampling step.
    During eval() the mean mu is used directly (deterministic reconstruction).
    """
    def __init__(self, latent_dim=32):
        super().__init__()
        self.latent_dim = latent_dim

        # Shared encoder backbone
        self.enc_backbone = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1),   # → 32×14×14
            nn.BatchNorm2d(32), nn.LeakyReLU(0.2),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),  # → 64×7×7
            nn.BatchNorm2d(64), nn.LeakyReLU(0.2),
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 256), nn.LeakyReLU(0.2),
        )
        self.fc_mu     = nn.Linear(256, latent_dim)
        self.fc_logvar = nn.Linear(256, latent_dim)

        # Decoder
        self.fc_decode = nn.Sequential(
            nn.Linear(latent_dim, 256), nn.LeakyReLU(0.2),
            nn.Linear(256, 64 * 7 * 7), nn.LeakyReLU(0.2),
            nn.Unflatten(1, (64, 7, 7)),
        )
        self.dec_conv = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(32), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(32,  1, 3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid(),
        )

    def encode(self, x):
        h = self.enc_backbone(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        if self.training:
            std = torch.exp(0.5 * logvar)
            return mu + std * torch.randn_like(std)
        return mu   # deterministic at eval time

    def decode(self, z):
        return self.dec_conv(self.fc_decode(z))

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

    def sample(self, n, device):
        """Generate n images by sampling z ~ N(0,I)."""
        self.eval()
        z = torch.randn(n, self.latent_dim).to(device)
        with torch.no_grad():
            return self.decode(z)


def vae_loss(recon, x, mu, logvar, beta=1.0):
    """ELBO = Reconstruction MSE + beta * KL(q(z|x) || N(0,I))."""
    batch_size = x.size(0)
    recon_loss = F.mse_loss(recon, x, reduction='sum') / batch_size
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / batch_size
    return recon_loss + beta * kl, recon_loss, kl


vae = ConvVAE(latent_dim=32).to(device)
print(f'VAE parameters: {sum(p.numel() for p in vae.parameters()):,}')

_x = torch.randn(4, 1, 28, 28).to(device)
_r, _mu, _lv = vae(_x)
print(f'Input: {_x.shape} | mu: {_mu.shape} | logvar: {_lv.shape} | Output: {_r.shape}')
del _x, _r, _mu, _lv


In [ ]:
vae_optimizer = optim.Adam(vae.parameters(), lr=1e-3, weight_decay=1e-5)
vae_scheduler = optim.lr_scheduler.CosineAnnealingLR(vae_optimizer, T_max=30)

num_epochs_vae = 30
vae_hist = {k: [] for k in
            ['train_total', 'train_recon', 'train_kl',
             'val_total',   'val_recon',   'val_kl']}

for epoch in range(num_epochs_vae):
    # KL annealing: linearly ramp beta 0→1 over the first 50% of training
    # to avoid posterior collapse before the encoder learns to reconstruct.
    beta = min(1.0, (epoch + 1) / (num_epochs_vae * 0.5))

    # ---- Train ----
    vae.train()
    t_tot = t_rec = t_kl = 0.0
    for data, _ in train_loader:
        data = data.to(device)
        recon, mu, logvar = vae(data)
        loss, rl, kl = vae_loss(recon, data, mu, logvar, beta)
        vae_optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(vae.parameters(), 1.0)
        vae_optimizer.step()
        t_tot += loss.item(); t_rec += rl.item(); t_kl += kl.item()

    # ---- Validate (always beta=1 to track true ELBO) ----
    vae.eval()
    v_tot = v_rec = v_kl = 0.0
    with torch.no_grad():
        for data, _ in test_loader:
            data = data.to(device)
            recon, mu, logvar = vae(data)
            loss, rl, kl = vae_loss(recon, data, mu, logvar, beta=1.0)
            v_tot += loss.item(); v_rec += rl.item(); v_kl += kl.item()

    vae_scheduler.step()

    n_tr, n_v = len(train_loader), len(test_loader)
    for key, val in zip(
        ['train_total', 'train_recon', 'train_kl', 'val_total', 'val_recon', 'val_kl'],
        [t_tot/n_tr, t_rec/n_tr, t_kl/n_tr, v_tot/n_v, v_rec/n_v, v_kl/n_v],
    ):
        vae_hist[key].append(val)

    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d}/{num_epochs_vae} [beta={beta:.2f}] | '
              f'Train {t_tot/n_tr:.4f} (R={t_rec/n_tr:.4f} KL={t_kl/n_tr:.4f}) | '
              f'Val {v_tot/n_v:.4f}')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, tk, vk, title in zip(
    axes,
    ['train_total', 'train_recon', 'train_kl'],
    ['val_total',   'val_recon',   'val_kl'],
    ['Total Loss (ELBO)', 'Reconstruction Loss', 'KL Divergence'],
):
    ax.plot(vae_hist[tk], label='Train')
    ax.plot(vae_hist[vk], label='Validation')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
ae.eval(); vae.eval()
test_batch = sample_imgs[:10].to(device)

with torch.no_grad():
    ae_r,  _       = ae(test_batch)
    vae_r, _, _    = vae(test_batch)
    ae_r  = ae_r.cpu()
    vae_r = vae_r.cpu()

fig, axes = plt.subplots(3, 10, figsize=(18, 6))
for i in range(10):
    axes[0, i].imshow(test_batch[i].cpu().squeeze(), cmap='gray'); axes[0, i].axis('off')
    axes[1, i].imshow(ae_r[i].squeeze(),            cmap='gray'); axes[1, i].axis('off')
    axes[2, i].imshow(vae_r[i].squeeze(),           cmap='gray'); axes[2, i].axis('off')
axes[0, 0].set_ylabel('Original', fontsize=11)
axes[1, 0].set_ylabel('AE',       fontsize=11)
axes[2, 0].set_ylabel('VAE',      fontsize=11)
plt.suptitle('Reconstruction: Autoencoder vs VAE', fontsize=13)
plt.tight_layout()
plt.show()

tb = test_batch.cpu()
ae_mse  = F.mse_loss(ae_r,  tb).item()
vae_mse = F.mse_loss(vae_r, tb).item()
print(f'AE  | MSE: {ae_mse:.5f}  PSNR: {10*np.log10(1/ae_mse):.2f} dB')
print(f'VAE | MSE: {vae_mse:.5f}  PSNR: {10*np.log10(1/vae_mse):.2f} dB')
print('Note: VAE may have slightly higher MSE due to KL regularisation,')
print('      but its latent space supports coherent generation.')


In [ ]:
vae.eval()

# ---- 1. Generate from prior N(0,I) ----
gen_samples = vae.sample(16, device).cpu()

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i in range(16):
    axes[i // 8, i % 8].imshow(gen_samples[i].squeeze(), cmap='gray')
    axes[i // 8, i % 8].axis('off')
plt.suptitle('VAE: Samples Generated from N(0,I)  →  Coherent Digits')
plt.tight_layout()
plt.show()

# ---- 2. Latent space structure (collect all test mus) ----
all_mu, all_labels = [], []
with torch.no_grad():
    for data, labels in test_loader:
        mu, _ = vae.encode(data.to(device))
        all_mu.append(mu.cpu())
        all_labels.append(labels)
all_mu     = torch.cat(all_mu)
all_labels = torch.cat(all_labels)

pca_vae  = PCA(n_components=2).fit_transform(all_mu.numpy())
z_range  = np.linspace(-4, 4, 300)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for c in range(10):
    mask = all_labels == c
    axes[0].scatter(pca_vae[mask, 0], pca_vae[mask, 1],
                    label=str(c), alpha=0.4, s=6)
axes[0].set_title('VAE Latent Space (PCA of mu)  —  Structured & Clustered')
axes[0].legend(markerscale=2, loc='best', title='Digit')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')

axes[1].hist(all_mu.numpy().flatten(), bins=100, density=True,
             alpha=0.7, label='VAE mu')
axes[1].plot(z_range, np.exp(-0.5 * z_range**2) / np.sqrt(2 * np.pi),
             'r--', lw=2, label='N(0,1)')
axes[1].set_title('VAE Latent Distribution  —  KL-Regularised')
axes[1].set_xlabel('Value')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'VAE latent | mean: {all_mu.mean():.3f} (target≈0)  '
      f'std: {all_mu.std():.3f} (target≈1)')


In [ ]:
vae.eval()

img_a = sample_imgs[0:1]   # first test image
img_b = sample_imgs[4:5]   # another test image

with torch.no_grad():
    mu_a, _ = vae.encode(img_a.to(device))
    mu_b, _ = vae.encode(img_b.to(device))

    alphas    = torch.linspace(0, 1, 10)
    interp_z  = torch.stack([(1 - a) * mu_a + a * mu_b for a in alphas]).squeeze(1)
    interp_vae = vae.decode(interp_z).cpu()

fig, axes = plt.subplots(1, 12, figsize=(18, 2.5))
axes[0].imshow(img_a.squeeze(), cmap='gray')
axes[0].set_title(f'Start\n({sample_labels[0].item()})')
axes[0].axis('off')
for i, img in enumerate(interp_vae):
    axes[i + 1].imshow(img.squeeze(), cmap='gray')
    axes[i + 1].set_title(f'{alphas[i]:.1f}')
    axes[i + 1].axis('off')
axes[11].imshow(img_b.squeeze(), cmap='gray')
axes[11].set_title(f'End\n({sample_labels[4].item()})')
axes[11].axis('off')
plt.suptitle('VAE: Smooth Latent Space Interpolation', y=1.08)
plt.tight_layout()
plt.show()


## Section 3: VAE for Medical Imaging

In [ ]:
IMG_SIZE   = 128
DATA_PATH  = './data/chest_xray_images'

class ChestXRayDataset(Dataset):
    """Loads all PNG/JPG images from a flat directory as grayscale tensors."""

    EXTS = {'.png', '.jpg', '.jpeg'}

    def __init__(self, root_dir, transform=None):
        self.root_dir  = root_dir
        self.transform = transform
        self.images    = [
            f for f in os.listdir(root_dir)
            if os.path.splitext(f)[1].lower() in self.EXTS
        ]
        if not self.images:
            raise RuntimeError(f'No images found in {root_dir}')

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        path  = os.path.join(self.root_dir, self.images[idx])
        image = Image.open(path).convert('L')   # grayscale
        if self.transform:
            image = self.transform(image)
        return image


xray_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),   # [0,1] → [-1,1] (matches tanh output)
])

xray_dataset = ChestXRayDataset(DATA_PATH, transform=xray_transform)
n_train = int(0.8 * len(xray_dataset))
n_val   = len(xray_dataset) - n_train
xray_train, xray_val = random_split(
    xray_dataset, [n_train, n_val],
    generator=torch.Generator().manual_seed(42),
)

xray_train_loader = DataLoader(xray_train, batch_size=32, shuffle=True,
                               pin_memory=True, num_workers=0)
xray_val_loader   = DataLoader(xray_val,   batch_size=32, shuffle=False,
                               pin_memory=True, num_workers=0)

print(f'Total: {len(xray_dataset)} | Train: {n_train} | Val: {n_val}')
print(f'Image size: {IMG_SIZE}×{IMG_SIZE}')
print(f'Batches — train: {len(xray_train_loader)}, val: {len(xray_val_loader)}')


In [ ]:
xray_batch = next(iter(xray_val_loader))

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i in range(10):
    img = (xray_batch[i] + 1) / 2   # denormalize to [0,1] for display
    axes[i // 5, i % 5].imshow(img.squeeze(), cmap='gray')
    axes[i // 5, i % 5].axis('off')
plt.suptitle(f'Chest X-Ray Sample Images ({IMG_SIZE}×{IMG_SIZE})')
plt.tight_layout()
plt.show()


In [ ]:
class ConvBlock(nn.Module):
    """Double-conv block: BN + LeakyReLU applied after each conv."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch), nn.LeakyReLU(0.2, inplace=True),
        )
    def forward(self, x): return self.block(x)


class VAEUNet(nn.Module):
    """Convolutional VAE with a U-Net style encoder/decoder for medical image generation.

    Encoder: 4× (ConvBlock → MaxPool2d), flattening to (mu, logvar).
    Decoder: FC → reshape → 4× (ConvTranspose2d → ConvBlock) → tanh output.

    img_size must be divisible by 16.
    """
    def __init__(self, in_channels=1, latent_dim=128, img_size=128):
        super().__init__()
        assert img_size % 16 == 0, 'img_size must be divisible by 16'
        self.latent_dim        = latent_dim
        self.img_size          = img_size
        self.bottleneck_hw     = img_size // 16    # spatial size at bottleneck (8 for 128)

        # ── Encoder ──────────────────────────────────────────────────────────
        self.enc1 = ConvBlock(in_channels, 32)    # → 32  × H    × W
        self.enc2 = ConvBlock(32,  64)            # → 64  × H/2  × W/2
        self.enc3 = ConvBlock(64,  128)           # → 128 × H/4  × W/4
        self.enc4 = ConvBlock(128, 256)           # → 256 × H/8  × W/8
        self.pool = nn.MaxPool2d(2, 2)

        flat = 256 * self.bottleneck_hw ** 2      # 256 × 8 × 8 = 16384 for img_size=128
        self.fc_mu     = nn.Linear(flat, latent_dim)
        self.fc_logvar = nn.Linear(flat, latent_dim)

        # ── Decoder ──────────────────────────────────────────────────────────
        self.fc_decode = nn.Linear(latent_dim, flat)

        self.up1  = nn.ConvTranspose2d(256, 128, 2, stride=2); self.dec1 = ConvBlock(128, 128)
        self.up2  = nn.ConvTranspose2d(128,  64, 2, stride=2); self.dec2 = ConvBlock(64,   64)
        self.up3  = nn.ConvTranspose2d(64,   32, 2, stride=2); self.dec3 = ConvBlock(32,   32)
        self.up4  = nn.ConvTranspose2d(32,   16, 2, stride=2); self.dec4 = ConvBlock(16,   16)

        self.out_conv = nn.Sequential(
            nn.Conv2d(16, in_channels, 1),
            nn.Tanh(),    # output in [-1, 1] — matches normalised input
        )

    def encode(self, x):
        x = self.pool(self.enc1(x))   # H   → H/2
        x = self.pool(self.enc2(x))   # H/2 → H/4
        x = self.pool(self.enc3(x))   # H/4 → H/8
        x = self.pool(self.enc4(x))   # H/8 → H/16
        x = x.view(x.size(0), -1)
        return self.fc_mu(x), self.fc_logvar(x)

    def reparameterize(self, mu, logvar):
        if self.training:
            std = torch.exp(0.5 * logvar)
            return mu + std * torch.randn_like(std)
        return mu

    def decode(self, z):
        x = F.leaky_relu(self.fc_decode(z), 0.2)
        x = x.view(x.size(0), 256, self.bottleneck_hw, self.bottleneck_hw)
        x = self.dec1(self.up1(x))
        x = self.dec2(self.up2(x))
        x = self.dec3(self.up3(x))
        x = self.dec4(self.up4(x))
        return self.out_conv(x)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

    def sample(self, n, device):
        self.eval()
        z = torch.randn(n, self.latent_dim).to(device)
        with torch.no_grad():
            return self.decode(z)


vae_xray = VAEUNet(in_channels=1, latent_dim=128, img_size=IMG_SIZE).to(device)
n_params = sum(p.numel() for p in vae_xray.parameters())
print(f'VAEUNet parameters:  {n_params:,}')
print(f'Latent dimension:    {vae_xray.latent_dim}')
print(f'Bottleneck spatial:  {vae_xray.bottleneck_hw}×{vae_xray.bottleneck_hw}')

_x = torch.randn(2, 1, IMG_SIZE, IMG_SIZE).to(device)
_r, _mu, _lv = vae_xray(_x)
print(f'Input: {_x.shape} | mu: {_mu.shape} | Output: {_r.shape}')
del _x, _r, _mu, _lv


In [ ]:
xray_optimizer = optim.Adam(vae_xray.parameters(), lr=1e-3, weight_decay=1e-5)
xray_scheduler = optim.lr_scheduler.CosineAnnealingLR(xray_optimizer, T_max=30)

num_epochs_xray = 30
xray_hist = {k: [] for k in
             ['train_total', 'train_recon', 'train_kl',
              'val_total',   'val_recon',   'val_kl']}
best_val_loss = float('inf')

for epoch in range(num_epochs_xray):
    beta = min(1.0, (epoch + 1) / (num_epochs_xray * 0.5))

    # ---- Train ----
    vae_xray.train()
    t_tot = t_rec = t_kl = 0.0
    for data in tqdm(xray_train_loader,
                     desc=f'Epoch {epoch+1:2d}/{num_epochs_xray}', leave=False):
        data = data.to(device)
        recon, mu, logvar = vae_xray(data)
        loss, rl, kl = vae_loss(recon, data, mu, logvar, beta)
        xray_optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(vae_xray.parameters(), 1.0)
        xray_optimizer.step()
        t_tot += loss.item(); t_rec += rl.item(); t_kl += kl.item()

    # ---- Validate ----
    vae_xray.eval()
    v_tot = v_rec = v_kl = 0.0
    with torch.no_grad():
        for data in xray_val_loader:
            data = data.to(device)
            recon, mu, logvar = vae_xray(data)
            loss, rl, kl = vae_loss(recon, data, mu, logvar, beta=1.0)
            v_tot += loss.item(); v_rec += rl.item(); v_kl += kl.item()

    xray_scheduler.step()

    n_tr, n_v = len(xray_train_loader), len(xray_val_loader)
    for key, val in zip(
        ['train_total', 'train_recon', 'train_kl', 'val_total', 'val_recon', 'val_kl'],
        [t_tot/n_tr, t_rec/n_tr, t_kl/n_tr, v_tot/n_v, v_rec/n_v, v_kl/n_v],
    ):
        xray_hist[key].append(val)

    if v_tot / n_v < best_val_loss:
        best_val_loss = v_tot / n_v
        torch.save(vae_xray.state_dict(), 'data/vae_xray_best.pth')

    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1:2d}/{num_epochs_xray} [beta={beta:.2f}] | '
              f'Train {t_tot/n_tr:.1f} (R={t_rec/n_tr:.1f} KL={t_kl/n_tr:.1f}) | '
              f'Val {v_tot/n_v:.1f}')

print(f'\nBest validation loss: {best_val_loss:.4f}')
vae_xray.load_state_dict(torch.load('data/vae_xray_best.pth', map_location=device))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, tk, vk, title in zip(
    axes,
    ['train_total', 'train_recon', 'train_kl'],
    ['val_total',   'val_recon',   'val_kl'],
    ['Total ELBO Loss', 'Reconstruction Loss', 'KL Divergence'],
):
    ax.plot(xray_hist[tk], label='Train')
    ax.plot(xray_hist[vk], label='Validation')
    ax.set_title(title); ax.set_xlabel('Epoch')
    ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
vae_xray.eval()
val_batch = next(iter(xray_val_loader)).to(device)

with torch.no_grad():
    recon_xray, _, _ = vae_xray(val_batch)
    orig_show  = ((val_batch[:8]    + 1) / 2).cpu().clamp(0, 1)
    recon_show = ((recon_xray[:8]   + 1) / 2).cpu().clamp(0, 1)

# Reconstruction grid
fig, axes = plt.subplots(2, 8, figsize=(16, 5))
for i in range(8):
    axes[0, i].imshow(orig_show[i].squeeze(),  cmap='gray'); axes[0, i].axis('off')
    axes[1, i].imshow(recon_show[i].squeeze(), cmap='gray'); axes[1, i].axis('off')
axes[0, 0].set_ylabel('Original',     fontsize=11)
axes[1, 0].set_ylabel('Reconstructed', fontsize=11)
plt.suptitle('Chest X-Ray VAE: Reconstruction Quality', fontsize=13)
plt.tight_layout()
plt.show()

# Random generation grid
gen_xray = vae_xray.sample(8, device)
gen_show = ((gen_xray + 1) / 2).cpu().clamp(0, 1)

fig, axes = plt.subplots(1, 8, figsize=(16, 3))
for i in range(8):
    axes[i].imshow(gen_show[i].squeeze(), cmap='gray')
    axes[i].axis('off')
plt.suptitle('Chest X-Ray VAE: Generated Samples from N(0,I)')
plt.tight_layout()
plt.show()


In [ ]:
vae_xray.eval()
val_imgs = next(iter(xray_val_loader))
img_a = val_imgs[0:1].to(device)
img_b = val_imgs[1:2].to(device)

with torch.no_grad():
    mu_a, _ = vae_xray.encode(img_a)
    mu_b, _ = vae_xray.encode(img_b)

    alphas     = torch.linspace(0, 1, 8)
    interp_z   = torch.stack([(1 - a) * mu_a + a * mu_b for a in alphas]).squeeze(1)
    interp_xray = vae_xray.decode(interp_z)
    interp_show = ((interp_xray + 1) / 2).cpu().clamp(0, 1)

fig, axes = plt.subplots(1, 10, figsize=(18, 3))
axes[0].imshow(((img_a[0] + 1) / 2).cpu().squeeze(), cmap='gray')
axes[0].set_title('A'); axes[0].axis('off')
for i in range(8):
    axes[i + 1].imshow(interp_show[i].squeeze(), cmap='gray')
    axes[i + 1].set_title(f'a={alphas[i]:.2f}')
    axes[i + 1].axis('off')
axes[9].imshow(((img_b[0] + 1) / 2).cpu().squeeze(), cmap='gray')
axes[9].set_title('B'); axes[9].axis('off')
plt.suptitle('Chest X-Ray VAE: Smooth Latent Interpolation (A → B)', y=1.05)
plt.tight_layout()
plt.show()

# ── Evaluation metrics over full validation set ────────────────────────────────
all_orig, all_recon = [], []
with torch.no_grad():
    for batch in xray_val_loader:
        batch = batch.to(device)
        recon, _, _ = vae_xray(batch)
        all_orig.append(batch.cpu())
        all_recon.append(recon.cpu())

all_orig  = torch.cat(all_orig)
all_recon = torch.cat(all_recon)

# Denormalise to [0,1] before computing metrics
orig_01  = (all_orig  + 1) / 2
recon_01 = (all_recon + 1) / 2

mse  = F.mse_loss(recon_01, orig_01).item()
mae  = F.l1_loss(recon_01,  orig_01).item()
psnr = 10 * np.log10(1.0 / mse)

print('=' * 40)
print('Chest X-Ray VAE — Evaluation Metrics')
print('=' * 40)
print(f'MSE:  {mse:.5f}')
print(f'MAE:  {mae:.5f}')
print(f'PSNR: {psnr:.2f} dB')
print(f'Latent dim:   {vae_xray.latent_dim}')
print(f'Image pixels: {IMG_SIZE**2:,}')
print(f'Compression:  {IMG_SIZE**2} → {vae_xray.latent_dim} '
      f'({(1 - vae_xray.latent_dim / IMG_SIZE**2) * 100:.1f}% reduction)')
